In [ ]:
import time
import pandas as pd

In [ ]:


X_test = pd.read_csv('data/preprocessedTest.csv')
benign = ['MQTT_Publish', 'Thing_Speak', 'Wipro_bulb']
y_test = X_test.pop('Attack_type')
y_test = pd.DataFrame([0 if x in benign else 1 for x in y_test]).squeeze()


sample = X_test.iloc[0:1]   # keeps it as a DataFrame with one row


In [ ]:

repeats = 2000
total = 0
model=bestRF
for _ in range(repeats):
    start = time.perf_counter()
    y = model.predict(sample)     # or model.predict_proba(sample)
    end = time.perf_counter()
    total += (end - start)

print(f"Avg ML CPU inference: {(total/repeats)*1000:.6f} ms")


In [ ]:
#dl

from BinaryCNN_classFile import BinaryCNN
def load_model(model_path, device):
    model = BinaryCNN()
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print("Model loaded successfully.")
    return model

cnn=load_model(r"1dcnn_binary.pth",'cpu')
x = np.load("../test_inputs.npy").astype(np.float32)
y = np.load("../test_labels.npy")


xtestpts, ytestpts = next(iter(test_loader))
print(xtestpts.shape, len(xtestpts))
xtestpts_0 = xtestpts[0]
print(xtestpts_0.shape)
xtestpts_0 =xtestpts_0.unsqueeze(1)
cnn.eval()
with torch.no_grad():
    output = cnn(xtestpts_0)

print(output, y[0])
import time
cnn.eval()
with torch.no_grad():
    total = 0
    for _ in range(2000):
        start = time.perf_counter()
        y = cnn(xtestpts_0)
        end = time.perf_counter()
        total += (end - start)

print(f"CPU avg: {(total/2000)*1000:.6f} ms")

In [ ]:
#convert to onnx model
# ml model (xgb)
import pickle

file_path = '../models/xgb_binary_model.pkl' 

with open(file_path, 'rb') as file:
    loaded_model = pickle.load(file)
    
import pickle
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
rf = bestRF

num_features = rf.n_features_in_
initial_type = [('input', FloatTensorType([None, num_features]))]

# Convert to ONNX
onnx_model = convert_sklearn(rf, initial_types=initial_type)

# Save ONNX
with open("rf_binary.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [ ]:
class BinaryCNN(nn.Module):
    def __init__(self, input_length=83, dropout_rate=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(1,3), padding=(0,1))
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(1,3), padding=(0,1))

        self.pool = nn.MaxPool2d(kernel_size=(1,2))
        self.dropout = nn.Dropout(0.3)
#        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * (input_length // 2), 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.unsqueeze(2)
        print(x.shape)
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.reshape(x.size(0),-1)

        x = F.relu(self.fc1(x))
        # x = self.fc2(x)
        x = self.fc2(self.dropout(x))

        return x
    
    
print("Done")

# Check if gpu is available and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")